In [ ]:
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/My Drive/Blockhouse/first_25000_rows.csv")


In [ ]:
df.head()
print(df.columns.tolist())


['ts_recv', 'ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side', 'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence', 'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_ct_00', 'ask_ct_00', 'bid_px_01', 'ask_px_01', 'bid_sz_01', 'ask_sz_01', 'bid_ct_01', 'ask_ct_01', 'bid_px_02', 'ask_px_02', 'bid_sz_02', 'ask_sz_02', 'bid_ct_02', 'ask_ct_02', 'bid_px_03', 'ask_px_03', 'bid_sz_03', 'ask_sz_03', 'bid_ct_03', 'ask_ct_03', 'bid_px_04', 'ask_px_04', 'bid_sz_04', 'ask_sz_04', 'bid_ct_04', 'ask_ct_04', 'bid_px_05', 'ask_px_05', 'bid_sz_05', 'ask_sz_05', 'bid_ct_05', 'ask_ct_05', 'bid_px_06', 'ask_px_06', 'bid_sz_06', 'ask_sz_06', 'bid_ct_06', 'ask_ct_06', 'bid_px_07', 'ask_px_07', 'bid_sz_07', 'ask_sz_07', 'bid_ct_07', 'ask_ct_07', 'bid_px_08', 'ask_px_08', 'bid_sz_08', 'ask_sz_08', 'bid_ct_08', 'ask_ct_08', 'bid_px_09', 'ask_px_09', 'bid_sz_09', 'ask_sz_09', 'bid_ct_09', 'ask_ct_09', 'symbol']


In [ ]:
print("Number of rows:", len(df))

Number of rows: 5000


In [ ]:
# Converting timestamps and sort
df['ts_event'] = pd.to_datetime(df['ts_event'])
df = df.sort_values('ts_event').reset_index(drop=True)

**Best-Level OFI Function**

In [ ]:
def compute_best_level_ofi(df):
    ofi_values = [0]  # First value is 0 since there's no prior data

    bid_price = df['bid_px_00']
    ask_price = df['ask_px_00']
    bid_size = df['bid_sz_00']
    ask_size = df['ask_sz_00']

    for i in range(1, len(df)):
        # Bid side logic
        if bid_price[i] > bid_price[i - 1]:
            bid_contrib = bid_size[i]
        elif bid_price[i] < bid_price[i - 1]:
            bid_contrib = -bid_size[i - 1]
        else:
            bid_contrib = bid_size[i] - bid_size[i - 1]

        # Ask side logic
        if ask_price[i] < ask_price[i - 1]:
            ask_contrib = -ask_size[i]
        elif ask_price[i] > ask_price[i - 1]:
            ask_contrib = ask_size[i - 1]
        else:
            ask_contrib = ask_size[i - 1] - ask_size[i]

        ofi = bid_contrib + ask_contrib
        ofi_values.append(ofi)

    return pd.DataFrame({
        'ts_event': df['ts_event'],
        'best_level_ofi': ofi_values
    })

# Run the function
best_level_ofi_df = compute_best_level_ofi(df)

In [ ]:
# Analyzing the first few rows
best_level_ofi_df.head()

,ts_event,best_level_ofi
0,2024-10-21 11:54:29.221064336+00:00,0
1,2024-10-21 11:54:29.223769812+00:00,2
2,2024-10-21 11:54:29.225030400+00:00,3
3,2024-10-21 11:54:29.712434212+00:00,0
4,2024-10-21 11:54:29.764673165+00:00,0


In [ ]:
best_level_ofi_df.to_csv('best_level_ofi_output.csv', index=False)


In [ ]:
from google.colab import files
files.download('best_level_ofi_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Starting with Multi-Level OFI**

**Multi-Level OFI Function**


In [ ]:
def compute_multi_level_ofi(df, levels=10):
    ofi_values = [0]  # First row has no prior value

    for i in range(1, len(df)):
        total_ofi = 0

        for lvl in range(levels):
            bid_px_col = f'bid_px_0{lvl}'
            ask_px_col = f'ask_px_0{lvl}'
            bid_sz_col = f'bid_sz_0{lvl}'
            ask_sz_col = f'ask_sz_0{lvl}'

            # Bid side
            if df[bid_px_col][i] > df[bid_px_col][i - 1]:
                bid_contrib = df[bid_sz_col][i]
            elif df[bid_px_col][i] < df[bid_px_col][i - 1]:
                bid_contrib = -df[bid_sz_col][i - 1]
            else:
                bid_contrib = df[bid_sz_col][i] - df[bid_sz_col][i - 1]

            # Ask side
            if df[ask_px_col][i] < df[ask_px_col][i - 1]:
                ask_contrib = -df[ask_sz_col][i]
            elif df[ask_px_col][i] > df[ask_px_col][i - 1]:
                ask_contrib = df[ask_sz_col][i - 1]
            else:
                ask_contrib = df[ask_sz_col][i - 1] - df[ask_sz_col][i]

            total_ofi += bid_contrib + ask_contrib

        ofi_values.append(total_ofi)

    return pd.DataFrame({
        'ts_event': df['ts_event'],
        'multi_level_ofi': ofi_values
    })

# Computing and saving Multi-Level OFI
multi_level_ofi_df = compute_multi_level_ofi(df, levels=10)
multi_level_ofi_df.to_csv('multi_level_ofi_output.csv', index=False)

In [ ]:
multi_level_ofi_df.head()

,ts_event,multi_level_ofi
0,2024-10-21 11:54:29.221064336+00:00,0
1,2024-10-21 11:54:29.223769812+00:00,2
2,2024-10-21 11:54:29.225030400+00:00,3
3,2024-10-21 11:54:29.712434212+00:00,200
4,2024-10-21 11:54:29.764673165+00:00,-200


In [ ]:
from google.colab import files
files.download('multi_level_ofi_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Integrated OFI**

In [ ]:
def compute_integrated_ofi(ofi_df, window_seconds=1):

    ofi_df = ofi_df.copy()
    ofi_df['ts_event'] = pd.to_datetime(ofi_df['ts_event'])
    ofi_df.set_index('ts_event', inplace=True)

    integrated = ofi_df['best_level_ofi'].rolling(f'{window_seconds}s').sum()

    return integrated.reset_index().rename(columns={'best_level_ofi': f'integrated_ofi_{window_seconds}s'})

In [ ]:
# Compute Integrated OFI over 1-second window
integrated_ofi_df = compute_integrated_ofi(best_level_ofi_df, window_seconds=1)

# Saving it to CSV
integrated_ofi_df.to_csv('integrated_ofi_output.csv', index=False)

# Downloading the csv file
from google.colab import files
files.download('integrated_ofi_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Cross-Asset OFI**

Since the dataset contains data for only a single security (AAPL), it is not feasible to directly compute Cross-Asset OFI, which requires synchronized order flow information across multiple assets.